In [3]:
from selenium import webdriver

from selenium.webdriver.common.by import By

from selenium.webdriver.common.keys import Keys
import pandas as pd
from sqlalchemy import create_engine

In [11]:
# 웹드라이버 이용하여 구글크롬 브라우저 오픈

driver = webdriver.Chrome()

In [12]:
#특정 주소로 요청을 보낸다
driver.get('http://www.naver.com')

In [13]:
# Tag 중 id 가 query인 태그를 찾는법
search_element = \
    driver.find_element(By.ID, 'query')


In [14]:
type(search_element)

selenium.webdriver.remote.webelement.WebElement

In [15]:
# search_element(검색어창)에 특정 텍스트를
# 입력(데이터를 보낸다)한다.
search_element.send_keys('아이폰')

In [16]:
search_element.send_keys(Keys.ENTER)

In [17]:
# 쇼핑 버튼(Tag)를 선택한다
# LINK_TEXT가 쇼핑인 태크 
# -> 해당 조건에 맞는 태그가 몇개인가?
len(driver.find_elements(By.LINK_TEXT, '쇼핑'))
# 조건에 맞는 태그의 개수가 1개 이므로 클릭 이벤트 발생
driver.find_element(By.LINK_TEXT, '쇼핑').click()

In [123]:
# 쇼핑 페이지에서 검색어의 결과(물건의 가격과 제품명)를
# 크롤링 -> 해당 페이제의 소스코드를 불러온다.
html_data = driver.page_source

In [124]:
# bs4 안에 있는 BeautifulSoup 을 이용하여
# html_data 를 parsing(데이터의 파일을 변환)
from bs4 import BeautifulSoup as bs

In [125]:
soup = bs(html_data, 'html.parser')

In [126]:
# html_data가 2개의 탭 중 어떤 소스코드인지 확인
# 탭의 제목이 다르기 때문에 title Tag를 확인
# 특정 태그를 확인
soup.title

<title>아이폰 : 네이버 검색</title>

In [127]:
# page_source가 첫번째 탭인것을 확인
# 두번째 탭으로 page_source를 이동
# 탭의 정보를 확인
# driver에서 탭의 주소를 확인
driver.window_handles

['6E62066AE203438400F964FC29066354', '8C155956100E0036F0F8BF0A4B43809C']

In [128]:
# driver에서 탭을 이동
driver.switch_to.window(
    driver.window_handles[1]
)

In [153]:
html_data2 = driver.page_source
soup2 = bs(html_data2, 'html.parser')
soup2.title

<title data-next-head="">아이폰 : 네이버 가격비교</title>

#### 네이버 쇼핑 크롤링
1. div 태그중 id가 'content'인 태그를 선택하여 저장
2. 저장한 태그에서 div 태그 중 class가
"product_item__KQayS" 인 태그 정보를 모두 저장
3. div_list 에서 div 태그 중 class 가 
'product_title__ljFM_' 인 태그의 문자를 추출(상품명)
4. div_list에서 span 태그 중 class 가 'price'인 태그의 문자를 추출(상품가격)

In [154]:
content_data = soup2.find(
    'div',
    attrs={
        'id' : 'content'

    }
)

In [135]:
import re

In [156]:
# content_data 에서 div중 class가 'product_item__KQayS' 인 태그의 개수를 확인
# class의 이름에 'product_item' 이 포함되어있는 태그 (in연산자, isin함수)
len(
    content_data.find_all(
        'div',
        # attrs={
        #     'class' : 'product_item__KQayS'
        # }
        attrs={
            'class' : re.compile('product_item')
        }
    )
)

3

In [157]:
div_list = content_data.find_all(
        'div',
        # attrs={
        #     'class' : 'product_item__KQayS'
        # }
        attrs={
            'class' : re.compile('product_item')
        }
    )

In [158]:
div_data = div_list[0]

In [159]:
item_name = div_data.find(
    'div',
    # attrs={
    #     'class' : 'product_title__ljFM_'
    # }
    attrs={
            'class' : re.compile('product_title')
        }
).get_text()
# get_text() 로 상품의 이름추출

In [160]:
item_price = div_data.find(
    'span',
    attrs={
        'class' : 'price'
    }
).get_text()

In [161]:
# 상품의 이름과 상품의 가격을 dict 형태로 생성

# 비어있는 dict 생성
dict_data = {}

In [162]:
# dict에 데이터를 추가
dict_data['상품명'] = item_name
dict_data['가격'] = item_price

In [163]:
dict_data

{'상품명': 'Apple 애플 아이폰 13 미니 128GB 새상품', '가격': '298,000원'}

In [164]:
values = []
for div_data in div_list :
    item_name = div_data.find(
        'div',
        attrs={
                'class' : re.compile('product_title')
            }
    ).get_text()
    # get_text() 로 상품의 이름추출
    item_price = div_data.find(
        'span',
        attrs={
            'class' : 'price'
    }
    ).get_text()

    item_url = div_data.find('a')['href']
    dict_data = {
        '상품명' : item_name,
        '가격' : item_price,
        'url' : item_url
    }
    values.append(dict_data)

In [165]:
dict_data

{'상품명': '아이폰 16 프로 256GB [자급제]',
 '가격': '최저1,647,990원',
 'url': 'https://cr.shopping.naver.com/adcr?x=4uRq576EYnwd3uhyaGcLr%2F%2F%2F%2Fw%3D%3Ds%2BPyRoZNKxGQuyfSsZtPO57pmVY3Xey03tOz%2FKYQb9%2BPxTAJR5s3V3RheLYZrm3hTOBaBhF5l%2BvA5bAFNyi9KOGg6HabDTRx466CGciRiUnFBZuMfzSBinsMMtj7lqMfUBVGhmVHML%2FoOi9itZSLTOhQjQYN0tKjxsirlmOZJBTZcQGhct8neV0mpbM1%2F%2FjUn334rL%2BbR36BhYe0R%2FrUmjkI1yUrC8vID%2FKz%2F8c89qMuf0NR%2Fx5rWTcVMnKn8BCOoY2ZxuAuvSW7ZwIsaDJftz694IFlqpi2PXO88uXmhtpFBioyGYc1IlucKuBLuPxSHErLcuhiXatVqO87FsX2HsLngnxQ7ssMZd09T0OjyjIxearTBvydD1LUI3JWpCADI6Kxc4kvmlr8UQdcqFMDHL2ovNxFWzA781dOsDXmXgY1sXg5lUvZD9zC7Fi9WJJOHHQAtCGRK9iKFQv9MeFFdC5w9%2FDMX2S1ZzF2txO07wpSHt5FeG%2FH7Xww%2F%2FMuALSolwsGcIUHBW8vQzAW7mNYnV3UjwQMKHmfZEGs9R%2F5gzgF6r3a9g44t9OmoRuqLb%2FN8sJ6iAi89TSRt%2BkoRk3XbsM6514s5mNZF3qb5pjETi4RP4PMPHCnudPTZjGRUD3iU27A%2Fy9Wcz7EU72Pz6EEGPV%2FhYbD8yW4Cq%2BfsSwu4xY5jtEEnHZo%2B%2FVuCJHvsmHjrDVCYCaeglF2rgr1u2OQIzQ%3D%3D&nvMid=53599506392&catId=50001519'}

In [166]:
values

[{'상품명': 'Apple 애플 아이폰 13 미니 128GB 새상품',
  '가격': '298,000원',
  'url': 'https://cr.shopping.naver.com/adcr?x=H3jI47xzlVQF1WYZmplp3%2F%2F%2F%2Fw%3D%3DsYntwomEJ0nqvVg9raFxyma1odJ2%2F6LBtogIwFjN3ri1yoYErd7Wx1dfdVO6hh7qV83GQjzL5%2BvkE3MMk6qyEHMKdlAuo4CRofAy%2FiHp8jqv8RtWXQ%2B4yWv%2FzzZujOpojKy%2Bx4HIUKHP69GhQQJYy9DA%2FDVJeNF5URVhhGNG3NQb8Ue6CMS5M75pT5vMuG7BKyaxxRm4H8HcYDi0B3B3iQmfQT7uWwvxdz8nMvhXCKF1zPnBo0jGi%2FSdSlDWdPE2TZznAAH%2FDJkgOxgHmnSf1jTForf4nGUgPwiv53t%2BtlmqAbCY81f975eS5nfR25FBiP83WAv8GAIoQ0MwLPWeb%2FQo3SNe5qLoqXVtULiXzi5p4PtM3NiCk0nB6XIa249O3G1fUqIZ7CTqeNBeEUiuIjRZhILDd3GYCAhDGtTHHbmXN81WjUpxJVJrN5mDsz4BpBWxMcPavFhCT3ZiddWGpiXDk%2FkbHZxyMzTBpfIN1hTOBbCOjR8RqdXREfQQSPpXKbOJEpyFE%2Ft7cEbAgiUS3%2BSs3Ydyg6Og7ejXW5W7UF8KeCV9X781%2Bj%2F1qryCSqaHqib5Y8cu9Y2%2B2oTPyG7FK%2BH4oxPDPvikdk1JPvM43tH7f7iYwg26TFva1BhhPw7DoA35CyMrpXQUt3%2FJvHupy5s1ITc7NlRPvorKrNyQBKaY%3D&nvMid=88573763568&catId=50001519'},
 {'상품명': '아이폰 16 프로 128GB [자급제]',
  '가격': '최저1,538,900원',
  'url': 'https://cr.shopping.na

In [167]:
# div_list에서 div 태그중 class의 이름에 'product_img_area'가 포함된 태그를 찾는다.
# 해당 태그에서 img 태그에 있는 속성의 값을 추출

div_data = div_list[0]

In [168]:
div_data

<div class="product_item__KQayS"><div class="product_inner__O9F_g"><div class="product_img_area__ziVZA"><div class="thumbnail_thumb_wrap__X013_ _wrapper"><a class="thumbnail_thumb__MG0r2 linkAnchor _nlog_click _nlog_impression_element" data-shp-area="lst*N.img" data-shp-area-dtl='[{"key":"trtr","value":"slsl"}]' data-shp-area-id="img" data-shp-area-type="slot" data-shp-contents-dtl='[{"key":"prod_nm","value":"Apple 애플 아이폰 13 미니 128GB 새상품"},{"key":"price","value":"298000"},{"key":"brand_seq","value":"144225"},{"key":"nfa_type","value":"NORMAL"},{"key":"catalog_nv_mid","value":"88573763568"},{"key":"organic_expose_order","value":"1"},{"key":"chnl_prod_no","value":"11029257361"},{"key":"std_yn","value":"n"}]' data-shp-contents-grp="prod" data-shp-contents-id="88573763568" data-shp-contents-provider-dtl='[{"key":"adsr_type","value":"SHOPN"},{"key":"brandstore_type","value":"n"},{"key":"chnl_no","value":"102459197"}]' data-shp-contents-provider-id="11501384" data-shp-contents-provider-type=

In [169]:
# div_data에서 a태그(하이퍼링크) 첫번째 정보를 확인하여 href 속성의 값을 출력
# html문서에서 첫번째 태그의 정보를 확인
    # htmldata.태그명
    # htmldata.find(태그명)
# 특정 태그의
div_data.a['href']
div_data.find('a')['href']

'https://cr.shopping.naver.com/adcr?x=H3jI47xzlVQF1WYZmplp3%2F%2F%2F%2Fw%3D%3DsYntwomEJ0nqvVg9raFxyma1odJ2%2F6LBtogIwFjN3ri1yoYErd7Wx1dfdVO6hh7qV83GQjzL5%2BvkE3MMk6qyEHMKdlAuo4CRofAy%2FiHp8jqv8RtWXQ%2B4yWv%2FzzZujOpojKy%2Bx4HIUKHP69GhQQJYy9DA%2FDVJeNF5URVhhGNG3NQb8Ue6CMS5M75pT5vMuG7BKyaxxRm4H8HcYDi0B3B3iQmfQT7uWwvxdz8nMvhXCKF1zPnBo0jGi%2FSdSlDWdPE2TZznAAH%2FDJkgOxgHmnSf1jTForf4nGUgPwiv53t%2BtlmqAbCY81f975eS5nfR25FBiP83WAv8GAIoQ0MwLPWeb%2FQo3SNe5qLoqXVtULiXzi5p4PtM3NiCk0nB6XIa249O3G1fUqIZ7CTqeNBeEUiuIjRZhILDd3GYCAhDGtTHHbmXN81WjUpxJVJrN5mDsz4BpBWxMcPavFhCT3ZiddWGpiXDk%2FkbHZxyMzTBpfIN1hTOBbCOjR8RqdXREfQQSPpXKbOJEpyFE%2Ft7cEbAgiUS3%2BSs3Ydyg6Og7ejXW5W7UF8KeCV9X781%2Bj%2F1qryCSqaHqib5Y8cu9Y2%2B2oTPyG7FK%2BH4oxPDPvikdk1JPvM43tH7f7iYwg26TFva1BhhPw7DoA35CyMrpXQUt3%2FJvHupy5s1ITc7NlRPvorKrNyQBKaY%3D&nvMid=88573763568&catId=50001519'

In [ ]:
# driver 스크롤을 가장 밑으로 내린다. 
# driver.execute_script(
#     "window.scrollTo(0, document.body.scrollHeight);"
# )

In [176]:
# driver에서 스크롤을 일정 간격으로 내린다
driver.execute_script(
    "window.scrollBy(0,800);"   
)

In [177]:
soup3 = driver.page_source

soup = bs(html_data, 'html.parser')

In [ ]:

# soup = bs(html_data, 'html.parser')

# 광고 상품을 제외한 모든 상품의 이름과 가격 링크 주소를
# 2차원 데이터(리스트 안에 딕셔너리)로 생성

# 생성된 2차원 데이터를 데이터프레임으로 생성

# 검색어('아이폰')를 파일명으로 csv 파일을 생성하고
# 인덱스는 제외한다.


In [ ]:
# BeautifilSoup을 이용해서 데이터 파싱
soup4 = bs(soup3, 'html.parser')

In [186]:
soup4.title

<title data-next-head="">아이폰 : 네이버 가격비교</title>

In [ ]:
div_list = soup4.find_all(
    'div',
    attrs = {re.compile('product_item')}
)

In [192]:
values = []
for div_data in div_list :
    item_name = div_data.find(
        'div',
        attrs={
                'class' : re.compile('product_title')
            }
    ).get_text()
    # get_text() 로 상품의 이름추출
    item_price = div_data.find(
        'span',
        attrs={
            'class' : 'price'
    }
    ).get_text()

    item_url = div_data.find('a')['href']
    dict_data = {
        '상품명' : item_name,
        '가격' : item_price,
        'url' : item_url
    }
    values.append(dict_data)

In [ ]:
values

In [194]:
len(values)

40

생성된 2차원 데이터를 데이터프레임으로 생성

검색어('아이폰')를 파일명으로 csv 파일을 생성하고
인덱스는 제외한다.

In [205]:
df = pd.DataFrame(values)

In [207]:
df.to_csv('아이폰.csv', index=False)

In [ ]:
df

In [216]:
driver.quit()